# Markdown Hierarchy Parser

> Parse Markdown into heading-addressable sections while preserving each section's source text

In [ ]:
#| default_exp md_hier

In [ ]:
#| export
import re, zlib
from fastcore.utils import *

In [ ]:
from fastcore.test import *

Markdown documents already divide themselves into useful semantic units. `HeadingDict` preserves that hierarchy, so a long document can be inspected by section instead of searched and sliced as one flat string. Every node is a dictionary of child headings with the complete Markdown for that section in `.text`.

## Heading trees

`HeadingDict` supports normal dictionary lookup. `at()` follows a heading path in one call, `paths()` lists every available path, and `find()` returns a heading when its title occurs exactly once. Explicit paths are preferable when a title such as "Hooks" appears in more than one place.

In [ ]:
#| export
class HeadingDict(dict):
    "A dictionary of child headings with the section Markdown in `text`."
    def __init__(self, text='', start_line=1, *args, **kwargs):
        super().__init__(*args, **kwargs)
        self.text,self.start_line = text,start_line

    def paths(self):
        "List every heading path as a tuple."
        res = []
        for title,node in self.items():
            res.append((title,))
            res += [(title,*path) for path in node.paths()]
        return res

    def at(self, *path):
        "Return the node at `path`."
        node = self
        for title in path: node = node[title]
        return node

    def find(self, title):
        "Return the node whose heading is uniquely named `title`."
        paths = [path for path in self.paths() if path[-1] == title]
        if len(paths) != 1: raise KeyError(f"Expected one heading named {title!r}, found {len(paths)}: {paths}")
        return self.at(*paths[0])

    def refresh(self):
        "Fresh tree re-read from `path` (roots from `create_heading_dict_file` only)"
        return create_heading_dict_file(self.path)

    def view(self,
        nums=False, # Prefix lines with document-absolute line numbers, `lineno: `
        lnhashes=False # Prefix `lineno|hash|` exhash addresses instead; wins over `nums`
    ):
        "Section source with document-absolute line prefixes, so an editor can address the source file directly"
        lines = self.text.splitlines()
        if lnhashes: return PrettyString('\n'.join(f'{self.start_line+i}|{zlib.crc32(l.encode()) & 0xffff:04x}|{l}' for i,l in enumerate(lines)))
        if nums: return PrettyString('\n'.join(f'{self.start_line+i}: {l}' for i,l in enumerate(lines)))
        return self.text

In [ ]:
#| export
def create_heading_dict(text, rm_fenced=True):
    "Create a nested `HeadingDict` from Markdown headings."
    lines = text.splitlines()
    headings,fence = [],None
    for i,line in enumerate(lines):
        if match := re.match(r'^\s*(`{3,}|~{3,})', line):
            marker = match[1][0]
            if fence == marker: fence = None
            elif fence is None: fence = marker
            if rm_fenced: continue
        if fence and rm_fenced: continue
        if match := re.match(r'^(#{1,6})\s+(.+?)\s*#*\s*$', line):
            headings.append(dict(level=len(match[1]), title=match[2], line=i))

    for i,heading in enumerate(headings):
        end = next((h['line'] for h in headings[i+1:] if h['level'] <= heading['level']), len(lines))
        heading['text'] = '\n'.join(lines[heading['line']:end]).strip()

    result = HeadingDict(text)
    stack,levels = [result],[0]
    for heading in headings:
        while len(stack) > 1 and levels[-1] >= heading['level']:
            stack.pop()
            levels.pop()
        parent,title = stack[-1],heading['title']
        if title in parent: raise ValueError(f"Duplicate sibling heading {title!r}")
        parent[title] = node = HeadingDict(heading['text'], heading['line']+1)
        stack.append(node)
        levels.append(heading['level'])
    return result

In [ ]:
sample_md = r'''# Hooks

Shared hook behavior.

## Common input fields

Every hook receives `session_id`.

## Hooks

### SessionStart

Runs at thread start.

```python
# This is code, not a heading
```

### PostCompact

Runs after compaction.'''

result = create_heading_dict(sample_md)
result.paths()

[('Hooks',),
 ('Hooks', 'Common input fields'),
 ('Hooks', 'Hooks'),
 ('Hooks', 'Hooks', 'SessionStart'),
 ('Hooks', 'Hooks', 'PostCompact')]

Use `at()` when you know the path. The returned node includes its heading and all Markdown beneath it, up to the next heading at the same or a higher level.

In [ ]:
session = result.at('Hooks', 'Hooks', 'SessionStart')
session.text

'### SessionStart\n\nRuns at thread start.\n\n```python\n# This is code, not a heading\n```'

For non-left nodes, it shows the whole section including lower level:

In [ ]:
hooks = result.at('Hooks', 'Hooks')
hooks.text

'## Hooks\n\n### SessionStart\n\nRuns at thread start.\n\n```python\n# This is code, not a heading\n```\n\n### PostCompact\n\nRuns after compaction.'

`find()` is shorter when a title is unique anywhere in the document. It raises when the title is absent or ambiguous, so it cannot quietly select the wrong section.

In [ ]:
test_eq(result.find('SessionStart'), session)
test_fail(lambda: result.find('Hooks'), contains='found 2')

## Fenced code and duplicate headings

Markdown examples often contain lines that look like headings. Backtick and tilde fenced blocks are ignored by default, so `# This is code` does not appear in the heading tree.

In [ ]:
expected = r'''### SessionStart

Runs at thread start.

```python
# This is code, not a heading
```'''
test_eq(result.find('SessionStart').text, expected)
test_eq([path for path in result.paths() if path[-1] == 'This is code'], [])

### Duplicate siblings

A dictionary cannot represent two children with the same key. Rather than silently discard the first section, `create_heading_dict` raises for duplicate sibling headings. Repeated titles under different parents are fine and remain addressable by path.

In [ ]:
test_fail(lambda: create_heading_dict('# A\n## Same\n## Same'), contains='Duplicate sibling heading')
test_eq(result.paths()[-1], ('Hooks', 'Hooks', 'PostCompact'))

## Line-addressed views

Each node records `start_line`, the 1-based line number of its heading in the original document, and `view()` can prefix each line of the section with its absolute line number or with a `lineno|hash|` address. The hash is the same CRC-32 formula used by [exhash](https://github.com/AnswerDotAI/exhash) (`zlib.crc32(line) & 0xffff`), so when the parsed text came from a local file, the output of `view(lnhashes=True)` provides valid exhash addresses for editing that file directly.

In [ ]:
result.find('PostCompact').view(lnhashes=True)

`nums=True` shows plain line numbers instead; `lnhashes` wins when both are set, and with neither, `view()` is just `text`. A section starts at its own heading line, so numbering lines up with the full document.

In [ ]:
sec = result.find('PostCompact')
test_eq(sample_md.splitlines()[sec.start_line-1], '### PostCompact')
test_eq(sec.view(lnhashes=True).splitlines()[0], f"{sec.start_line}|{zlib.crc32(b'### PostCompact') & 0xffff:04x}|### PostCompact")
test_eq(sec.view(nums=True).splitlines()[0], f"{sec.start_line}: ### PostCompact")
test_eq(sec.view(nums=True, lnhashes=True), sec.view(lnhashes=True))
test_eq(sec.view(), sec.text)

## Files

In [ ]:
#| export
def create_heading_dict_file(path, rm_fenced=True):
    "Create a `HeadingDict` from the Markdown file at `path` (`~` ok), recording `path` for `refresh`"
    path = Path(path).expanduser()
    res = create_heading_dict(path.read_text(), rm_fenced)
    res.path = path
    return res

`create_heading_dict_file` reads a Markdown file (expanding a leading `~`) and records the path on the returned root, so the whole local-file workflow needs no raw file reads. Since a parsed tree persists while the file may change, addresses can go stale after an edit: `refresh()` re-reads the file and returns a fresh tree. A stale address fails loudly at exhash's hash check rather than editing the wrong line.

In [ ]:
p = Path('tmp_sample.md')
p.write_text(sample_md)
d = create_heading_dict_file('tmp_sample.md')
test_eq(d.find('PostCompact').view(lnhashes=True), result.find('PostCompact').view(lnhashes=True))
test_eq(d.refresh().paths(), d.paths())
p.unlink()